Notes-
- Go through analysis procedures used for humans and check which apply to mice
- Check the new mouse data- mouse 28-30
- Force change and learning analysis parameters-
    - TTR
    - Range of applied force: expected to become more specified as the session progresses. 
    - Submovement analysis (force)- what parameters apply?
    - Force direction specificity/preference: does is change/is the learning easily changed for this? 
- What would trajectory changes look like in this case? 
- Touch-no touch epoch observation in mice data? 
- Saturation of data observed?  
- Pose tracking observations- labelling

Calculations for within trial variability-
- instaneous force magnitude
- change in instantaneous force magnitude- indvidually across one each vector, or cumulative across both vectors? 
- Instantaneous angular variability- across each vector or cumulative? 

In [11]:
import datajoint as dj
dj.config.load('/home/coder/project/Secrets/dj_local_conf.json')
dj.conn() 
direction_dict = {'LR':'Left - Right',
                  'RL':'Right - Left',
                  'AP':'Anterior - Posterior',
                  'PA':'Posterior - Anterior'}
#%% import necessary packages
import matplotlib.pyplot as plt
import pandas as pd
from ndnf_pipeline import lab, experiment, behavior_analysis
import numpy as np


[2026-09-09 09:11:05,341][INFO]: DataJoint is configured from /home/coder/project/Secrets/dj_local_conf.json


In [ ]:
# ── Diagnostics: where does the M013 pipeline stop? ──────────────────────────
subject_id     = 'M036'
touch_param_id = 0
sat_param_id   = 0

n_force_traces = len(experiment.TrialForceTrace & {'subject_id': subject_id})
n_saturation   = len(behavior_analysis.TrialSaturationTimes
                      & {'subject_id': subject_id, 'sat_param_id': sat_param_id})
n_touch        = len(behavior_analysis.TrialTouchTimes
                      & {'subject_id': subject_id, 'touch_param_id': touch_param_id,
                         'sat_param_id': sat_param_id})
n_touch_epochs = len(behavior_analysis.TrialTouchTimes.TouchEpoch
                      & {'subject_id': subject_id, 'touch_param_id': touch_param_id,
                         'sat_param_id': sat_param_id})

print(f'{subject_id}:')
print(f'  TrialForceTrace rows:            {n_force_traces}')
print(f'  TrialSaturationTimes rows:       {n_saturation}  (sat_param_id={sat_param_id})')
print(f'  TrialTouchTimes rows:            {n_touch}  (touch_param_id={touch_param_id})')
print(f'  TrialTouchTimes.TouchEpoch rows: {n_touch_epochs}')

if n_force_traces == 0:
    print('\n-> No raw force traces ingested for this subject yet.')
elif n_saturation == 0:
    print('\n-> Raw traces exist, but TrialSaturationTimes has not been populated yet.')
elif n_touch == 0:
    print('\n-> TrialSaturationTimes is populated, but TrialTouchTimes has not been populated yet.')
else:
    print('\n-> Touch epochs exist — if the earlier cell was still empty, check its subject_id/touch_param_id match these.')

In [ ]:
# ── Populate touch-epoch pipeline for M013 ────────────────────────────────
# NOTE: this writes new rows to the shared database. Run the diagnostics cell
# above first to confirm where the gap actually is.
subject_id = 'M036'

behavior_analysis.TrialSaturationTimes.populate(
    {'subject_id': subject_id}, display_progress=True, suppress_errors=True
)
behavior_analysis.TrialTouchTimes.populate(
    {'subject_id': subject_id}, display_progress=True, suppress_errors=True
)

In [ ]:
# ── Overview: sessions, blocks, and trials available for a subject ────────────
import numpy as np
import pandas as pd

subject_id = 'M036'

sessions = np.sort((experiment.Session & {'subject_id': subject_id}).fetch('session'))

if len(sessions) == 0:
    print(f'No sessions found for {subject_id}.')
else:
    rows = []
    for session in sessions:
        block_ids, feedback_types = (
            experiment.Block & {'subject_id': subject_id, 'session': int(session)}
        ).fetch('block', 'feedback_type', order_by='block')

        for block, feedback_type in zip(block_ids, feedback_types):
            trials = np.sort((experiment.BehaviorTrial
                               & {'subject_id': subject_id, 'session': int(session), 'block': int(block)}
                              ).fetch('trial'))
            rows.append({
                'session':       int(session),
                'block':         int(block),
                'feedback_type': feedback_type,
                'n_trials':      len(trials),
                'first_trial':   int(trials[0]) if len(trials) else None,
                'last_trial':    int(trials[-1]) if len(trials) else None,
                'trials':        trials.tolist(),
            })

    overview = pd.DataFrame(rows)
    print(f'{subject_id}: {overview["session"].nunique()} session(s), {len(overview)} block(s) total\n')
    print(overview.drop(columns='trials').to_string(index=False))
    # full per-trial numbers are still available in overview['trials'], e.g.:
    #   overview.loc[(overview.session == 8) & (overview.block == 0), 'trials'].iloc[0]

In [ ]:
# find the mouse IDs that have been used in the experiment and count the number of trials for each mouse, then plot a bar graph
mouse_IDs = lab.Subject.fetch('subject_id')
trial_nums = []
for m in mouse_IDs:
    trial_nums.append(len((experiment.SessionTrial()&{'subject_id':m})))
trial_nums = np.array(trial_nums)
needed_mice = trial_nums>0

fig = plt. figure()
plt.bar(mouse_IDs[needed_mice], trial_nums[needed_mice])
plt.xticks(rotation=45, ha="right")
plt.xlabel('Mouse ID')
plt.ylabel('Number of Trials')


In [ ]:

#subject_id = 'M004'#'mouse_bbenjamin'#'BCI81'
subject_id = 'mouse_judith'
#subject_id = 'mouse_bbenjamin'
subject_id = 'M013'

available_sessions = (experiment.Session()&{'subject_id':subject_id}).fetch('session')
trial_nums = []
hit_rates = []
session_lengths = []
for session in available_sessions:
    trials = (experiment.SessionTrial()&{'subject_id':subject_id,
                                         'session':session}).fetch('trial')
    trial_nums.append(len(trials))
    hits = len(experiment.BehaviorTrial()&{'subject_id':subject_id,'session':session,'outcome':'hit'})
    hit_rate = hits/len(trials) if len(trials)>0 else np.nan
    hit_rates.append(hit_rate)
    session_length = (experiment.SessionTrial()&{'subject_id':subject_id,'session':session,'trial':np.max(trials)}).fetch1('trial_end_time')
    session_lengths.append(session_length)
fig = plt.figure()
ax1 = fig.add_subplot(3,1,1)
ax1.plot(available_sessions,trial_nums,'o-')
ax1.set_xlabel('session')
ax1.set_ylabel('number of trials')
plt.title(f'{subject_id} behavior overview')
ax2 = fig.add_subplot(3,1,2)
ax2.plot(available_sessions,session_lengths,'o-')
ax2.set_xlabel('session')
ax2.set_ylabel('session length (s)')
ax3 = fig.add_subplot(3,1,3)
ax3.plot(available_sessions,hit_rates,'o-')
ax3.set_xlabel('session')
ax3.set_ylabel('hit rate')
plt.tight_layout()
plt.show()

In [ ]:
# visualize targets, and how forces change over sessions and trials
import matplotlib.pyplot as plt

# ── parameters ───────────────────────────────────────────────────────────────
subject_id     = 'M035'  # subject to visualize
session        = None       # int for a single session; None = all sessions
blocks_to_show = None        # None = all blocks; or e.g. [0, 1, 4]

subtract_force_median = False
force_uniform_range = True
uniform_force_range= np.asarray([-1,1,-1,1])*20

if session is None:
    _sess = sorted(set(
        (experiment.Block() & {'subject_id': subject_id})
        .fetch('session').tolist()))
else:
    _sess = [session]

for session_i, session in enumerate(_sess):
    available_blocks,feedback_types = (experiment.Block()&{'subject_id':subject_id,'session':session}).fetch('block','feedback_type')
    force_axes = (experiment.TaskSettings.ForceAxis()&{'subject_id':subject_id,'session':session}).fetch(as_dict=True)
    
    for block,feedback_type in zip(available_blocks,feedback_types):
        if blocks_to_show is not None and int(block) not in blocks_to_show:
            continue
        task_setting_id,target_force_lut = (experiment.TaskSettings()*experiment.Block() &{'subject_id':subject_id,'session':session,'block':block}).fetch1('task_setting_id','target_force_lut')
        
        fig = plt.figure(figsize=(8,8))
        ax_target_LUT = fig.add_subplot(2,2,1)
        force_axes_dict = {}
        for fa in force_axes:
            if fa['task_setting_id'] == task_setting_id:
                force_axes_dict[fa['force_axis_idx']] = {'force_direction':fa['force_direction'],
                                                    'target_force_axes':fa['target_force_axes']}
        if force_uniform_range:
            im0 = plt.imshow(np.ones(target_force_lut.shape)*np.min(target_force_lut.flatten()),extent = uniform_force_range,alpha = 1)                                            
        im_lut = plt.imshow(target_force_lut,extent = [force_axes_dict[0]['target_force_axes'][0],force_axes_dict[0]['target_force_axes'][-1],
                                                    force_axes_dict[1]['target_force_axes'][0],force_axes_dict[1]['target_force_axes'][-1]],
                                                    origin='upper',
                                                    cmap='viridis',
                                                    aspect='auto')
        plt.xlabel(direction_dict[force_axes_dict[0]['force_direction']])
        plt.ylabel(direction_dict[force_axes_dict[1]['force_direction']])
        plt.colorbar(im_lut,label='reward port speed')
        plt.title(f'{subject_id} s{session} b{block}: {feedback_type}')
        if force_uniform_range:
            ax_target_LUT.set_xlim([uniform_force_range[0],uniform_force_range[1]])
            ax_target_LUT.set_ylim([uniform_force_range[2],uniform_force_range[3]])


        ax_performance = fig.add_subplot(2,2,3)
        rewarded_trial, time_to_reward = (experiment.TrialEvent()*experiment.BehaviorTrial()*experiment.Block()&{'subject_id':subject_id,'session':session,'trial_event_type':'threshold crossing','block':block}).fetch('trial','trial_event_time')    #break
        ax_performance.set_xlabel('trial#')
        ax_performance.plot(rewarded_trial,time_to_reward,'g.')
        if len(rewarded_trial)>10:
            ax_performance.plot(np.convolve(rewarded_trial,np.ones(10)/10,mode='valid'),
                                np.convolve(np.asarray(time_to_reward,float),np.ones(10)/10,mode='valid'),'g-')
        
        ax_performance.set_ylabel('time to get reward')
        
        ax_force_hist = fig.add_subplot(2,2,2)
        force_traces_0_ = (experiment.TrialForceTrace.TrialForceAxis()*experiment.BehaviorTrial()*experiment.Block()&{'subject_id':subject_id,'session':session,'force_axis_idx':0,'block':block}).fetch('force_trace_value')
        force_traces_1_ = (experiment.TrialForceTrace.TrialForceAxis()*experiment.BehaviorTrial()*experiment.Block() &{'subject_id':subject_id,'session':session,'force_axis_idx':1,'block':block}).fetch('force_trace_value')
        if subtract_force_median:
            f0_baseline = np.median(np.concatenate(force_traces_0_))
            f1_baseline = np.median(np.concatenate(force_traces_1_))
            force_traces_0 = []
            for f in  force_traces_0_:
                force_traces_0.append(f-f0_baseline)
            force_traces_1 = []
            for f in  force_traces_1_:
                force_traces_1.append(f-f1_baseline)
        else:
            force_traces_0 = force_traces_0_
            force_traces_1 = force_traces_1_
        

        if force_uniform_range:
            histrange = [uniform_force_range[:2],uniform_force_range[:2]]
        else:
            histrange = [[force_axes_dict[0]['target_force_axes'][0],
                        force_axes_dict[0]['target_force_axes'][-1]],
                        [force_axes_dict[1]['target_force_axes'][0],
                        force_axes_dict[1]['target_force_axes'][-1]]]
        
        try:
            forcehist,binx,biny = np.histogram2d(np.concatenate(force_traces_0),
                                                np.concatenate(force_traces_1),
                                                range=histrange,
                                                bins=50)
            forcehist_all = forcehist.copy()
            forcehist = np.log(forcehist/sum(forcehist.flatten()))
            forcehist_ = forcehist.copy()
            forcehist_ [np.isinf(forcehist )] = 0
            forcehist[np.isinf(forcehist)] = np.nanmin(forcehist_.flatten())
            im_hist = ax_force_hist.imshow(forcehist,
                                        extent = [binx[0],binx[-1],biny[0],biny[-1]],
                                            alpha = 1)
            plt.colorbar(im_hist,label = 'fraction of time spent')
        except:
            pass

        plt.title('Force distribution')

        ax_force_hist_1d = fig.add_subplot(2,2,4)
        ax_force_hist_1d_2 = ax_force_hist_1d.twinx()
        try:
            ax_force_hist_1d.hist(np.concatenate(force_traces_0),binx,color = 'red',alpha = .5,label = 'force distribution ax 0')
            ax_force_hist_1d_2.hist(np.concatenate(force_traces_1),biny,color = 'blue',alpha = .5,label = 'force distribution ax 1')
            plt.legend()
            plt.xlabel('Force values')
        except:
            pass
        #ax_performance.set_yscale('log')
    

In [ ]:
# ── Per-trial TTR, force-magnitude change, and force-direction change ────────
# for specific sessions and blocks
import matplotlib.pyplot as plt
import pandas as pd

# ── parameters ───────────────────────────────────────────────────────────────
subject_id           = 'M035'
sessions_to_analyze  = [5,6,7,8]        # list of specific sessions, e.g. [8, 9]
blocks_to_analyze    = [1]       # list of specific blocks,  e.g. [0, 1]

rows = []
for session in sessions_to_analyze:
    for block in blocks_to_analyze:
        rewarded_trials, time_to_reward = (
            experiment.TrialEvent() * experiment.BehaviorTrial() * experiment.Block()
            & {'subject_id': subject_id, 'session': session, 'block': block,
               'trial_event_type': 'threshold crossing'}
        ).fetch('trial', 'trial_event_time', order_by='trial')

        for trial, ttr in zip(rewarded_trials, time_to_reward):
            trial = int(trial)
            trace_key = {'subject_id': subject_id, 'session': session, 'block': block, 'trial': trial}

            ax_idx, ax_val = (
                experiment.TrialForceTrace.TrialForceAxis()
                & trace_key & 'force_axis_idx < 2'
            ).fetch('force_axis_idx', 'force_trace_value', order_by='force_axis_idx')
            if len(ax_val) < 2:
                continue
            f0 = ax_val[list(ax_idx).index(0)].astype(float)
            f1 = ax_val[list(ax_idx).index(1)].astype(float)
            if len(f0) < 2 or len(f1) < 2:
                continue

            # instantaneous force magnitude and direction, cumulative across both vectors
            magnitude = np.sqrt(f0**2 + f1**2)
            direction = np.unwrap(np.arctan2(f1, f0))

            rows.append({
                'session':               session,
                'block':                 block,
                'trial':                 trial,
                'time_to_reward':        float(ttr),
                'delta_force_magnitude': float(magnitude[-1] - magnitude[0]),
                'delta_force_direction': float(direction[-1] - direction[0]),
            })

trial_metrics = pd.DataFrame(rows)
print(trial_metrics.to_string(index=False))

# ── plot per session/block ───────────────────────────────────────────────────
for (session, block), df_sb in trial_metrics.groupby(['session', 'block']):
    fig, axes = plt.subplots(3, 1, figsize=(8, 8), sharex=True)

    axes[0].plot(df_sb['trial'], df_sb['time_to_reward'], 'g.-')
    axes[0].set_ylabel('time to reward (s)')
    axes[0].set_title(f'{subject_id}  session {session}  block {block}')

    axes[1].plot(df_sb['trial'], df_sb['delta_force_magnitude'], '.-', color='tab:orange')
    axes[1].axhline(0, color='#999', lw=0.7, linestyle='--')
    axes[1].set_ylabel('Δ|F| per trial')

    axes[2].plot(df_sb['trial'], df_sb['delta_force_direction'], '.-', color='tab:blue')
    axes[2].axhline(0, color='#999', lw=0.7, linestyle='--')
    axes[2].set_ylabel('Δangle per trial (rad)')
    axes[2].set_xlabel('trial#')

    plt.tight_layout()
    plt.show()


In [ ]:
# ── Force-space trajectories per trial for specific subjects/sessions/blocks ──
import matplotlib.pyplot as plt
import numpy as np

# ── parameters ───────────────────────────────────────────────────────────────
subject_ids          = ['M035']   # list of specific subjects
sessions_to_analyze  = [5, 7, 8]  # list of specific sessions
blocks_to_analyze    = [1]        # list of specific blocks
n_cols               = 5          # trajectories per row in the grid

for subject_id in subject_ids:
    for session in sessions_to_analyze:
        for block in blocks_to_analyze:
            trials = np.sort((
                experiment.BehaviorTrial()
                & {'subject_id': subject_id, 'session': session, 'block': block}
            ).fetch('trial'))
            if len(trials) == 0:
                print(f'{subject_id} session {session} block {block}: no trials found')
                continue

            n_rows = int(np.ceil(len(trials) / n_cols))
            fig, axes = plt.subplots(n_rows, n_cols,
                                      figsize=(3 * n_cols, 3 * n_rows),
                                      squeeze=False)
            fig.suptitle(f'{subject_id}  session {session}  block {block}  '
                         '— force-space trajectory per trial')

            for i, trial in enumerate(trials):
                ax = axes[i // n_cols, i % n_cols]
                trial = int(trial)
                trace_key = {'subject_id': subject_id, 'session': session,
                             'block': block, 'trial': trial}

                ax_idx, ax_val = (
                    experiment.TrialForceTrace.TrialForceAxis()
                    & trace_key & 'force_axis_idx < 2'
                ).fetch('force_axis_idx', 'force_trace_value', order_by='force_axis_idx')
                if len(ax_val) < 2:
                    ax.set_visible(False)
                    continue
                f0 = ax_val[list(ax_idx).index(0)].astype(float)
                f1 = ax_val[list(ax_idx).index(1)].astype(float)

                ax.plot(f0, f1, color='#0077aa', lw=0.7, alpha=0.85)
                ax.plot(f0[0],  f1[0],  'o', color='lime', ms=5, zorder=3, label='start')
                ax.plot(f0[-1], f1[-1], 'o', color='red',  ms=5, zorder=3, label='end')
                ax.set_title(f'trial {trial}', fontsize=9)
                ax.set_xlabel('force axis 0', fontsize=7)
                ax.set_ylabel('force axis 1', fontsize=7)
                ax.tick_params(labelsize=6)

            # hide any unused grid cells
            for j in range(len(trials), n_rows * n_cols):
                axes[j // n_cols, j % n_cols].set_visible(False)

            handles, labels = axes[0, 0].get_legend_handles_labels()
            if handles:
                fig.legend(handles, labels, loc='lower center', ncol=2, fontsize=8,
                           bbox_to_anchor=(0.5, -0.01))
            plt.tight_layout()
            plt.show()
